### https://www.kaggle.com/competitions/drawing-with-llms

In [16]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [33]:
from transformers import AutoProcessor, AutoModel
import torch
from PIL import Image
import cairosvg
import os
import gc

class SVGMetricEvaluator:
    def __init__(self, model_name="google/siglip-so400m-patch14-384", device=None):
        # Initialize the device and model
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device is None else device
        print(f"Using device: {self.device}")
        
        # Load the model and processor
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.processor = AutoProcessor.from_pretrained(model_name)
    
    def svg_metric(self, prompt, svg):
        try:
            # Convert SVG to PNG
            cairosvg.svg2png(svg, write_to="./tmp/temp.png")
            
            # Open and process the image
            image = Image.open('./tmp/temp.png').convert("RGB")
            texts = ["SVG illustration of " + prompt]
            inputs = self.processor(text=texts, images=image, padding="max_length", return_tensors="pt").to(self.device)
            
            # Inference without gradient tracking
            with torch.no_grad():
                outputs = self.model(**inputs)
            
            logits_per_image = outputs.logits_per_image
            probs = torch.sigmoid(logits_per_image)
            
            # Clean up temporary PNG file
            os.remove('./tmp/temp.png')
            
            return probs[0][0].item()
        
        except Exception as e:
            print(f"An error occurred: {e}")
            return None
    
    def close_model(self):
        # Clean up to free memory
        del self.model
        gc.collect()

# Example of usage:
# evaluator = SVGMetricEvaluator()
# result = evaluator.svg_metric("some description", "<svg>...</svg>")
# print(result)


In [18]:
import concurrent
import io
import logging
import re
import re2

import cairosvg
import kagglehub
from lxml import etree
from vllm import LLM, SamplingParams
import torch
import multiprocessing
multiprocessing.set_start_method('spawn', force=True)
import gc

svg_constraints = kagglehub.package_import('metric/svg-constraints')
# ###/home/vino/.cache/kagglehub/notebooks/metric/svg-constraints/output/versions/1
# svg_metrics = kagglehub.package_import('jiazhuang/svg-image-fidelity/versions/12')
# ###/home/vino/.cache/kagglehub/notebooks/jiazhuang/svg-image-fidelity/output/versions/12


class SVGSanitizer:
    def __init__(self, constraints, default_svg):
        self.constraints = constraints
        self.default_svg = default_svg
    
    def enforce_constraints(self, svg_string: str) -> str:
        """Enforces constraints on an SVG string, removing disallowed elements
        and attributes.

        Parameters
        ----------
        svg_string : str
            The SVG string to process.

        Returns
        -------
        str
            The processed SVG string, or the default SVG if constraints
            cannot be satisfied.
        """
        logging.info('Sanitizing SVG...')

        try:
            parser = etree.XMLParser(remove_blank_text=True, remove_comments=True)
            root = etree.fromstring(svg_string, parser=parser)
        except etree.ParseError as e:
            logging.error('SVG Parse Error: %s. Returning default SVG.', e)
            return self.default_svg
    
        elements_to_remove = []
        for element in root.iter():
            tag_name = etree.QName(element.tag).localname
    
            # Remove disallowed elements
            if tag_name not in self.constraints.allowed_elements:
                elements_to_remove.append(element)
                continue  # Skip attribute checks for removed elements
    
            # Remove disallowed attributes
            attrs_to_remove = []
            for attr in element.attrib:
                attr_name = etree.QName(attr).localname
                if (
                    attr_name
                    not in self.constraints.allowed_elements[tag_name]
                    and attr_name
                    not in self.constraints.allowed_elements['common']
                ):
                    attrs_to_remove.append(attr)
    
            for attr in attrs_to_remove:
                logging.debug(
                    'Attribute "%s" for element "%s" not allowed. Removing.',
                    attr,
                    tag_name,
                )
                del element.attrib[attr]
    
            # Check and remove invalid href attributes
            for attr, value in element.attrib.items():
                 if etree.QName(attr).localname == 'href' and not value.startswith('#'):
                    logging.debug(
                        'Removing invalid href attribute in element "%s".', tag_name
                    )
                    del element.attrib[attr]

            # Validate path elements to help ensure SVG conversion
            if tag_name == 'path':
                d_attribute = element.get('d')
                if not d_attribute:
                    logging.warning('Path element is missing "d" attribute. Removing path.')
                    elements_to_remove.append(element)
                    continue # Skip further checks for this removed element
                # Use regex to validate 'd' attribute format
                path_regex = re2.compile(
                    r'^'  # Start of string
                    r'(?:'  # Non-capturing group for each command + numbers block
                    r'[MmZzLlHhVvCcSsQqTtAa]'  # Valid SVG path commands (adjusted to exclude extra letters)
                    r'\s*'  # Optional whitespace after command
                    r'(?:'  # Non-capturing group for optional numbers
                    r'-?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?'  # First number
                    r'(?:[\s,]+-?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?)*'  # Subsequent numbers with mandatory separator(s)
                    r')?'  # Numbers are optional (e.g. for Z command)
                    r'\s*'  # Optional whitespace after numbers/command block
                    r')+'  # One or more command blocks
                    r'\s*'  # Optional trailing whitespace
                    r'$'  # End of string
                )
                if not path_regex.match(d_attribute):
                    logging.warning(
                        'Path element has malformed "d" attribute format. Removing path.'
                    )
                    elements_to_remove.append(element)
                    continue
                logging.debug('Path element "d" attribute validated (regex check).')
        
        # Remove elements marked for removal
        for element in elements_to_remove:
            if element.getparent() is not None:
                element.getparent().remove(element)
                logging.debug('Removed element: %s', element.tag)

        try:
            cleaned_svg_string = etree.tostring(root, encoding='unicode')
            return cleaned_svg_string
        except ValueError as e:
            logging.error(
                'SVG could not be sanitized to meet constraints: %s', e
            )
            return self.default_svg

class SVGProcessor:
    @staticmethod
    def clean_and_extract_svgs(text, default_svg):
        text = re.sub(r'^.*?(<svg\b)', r'\1', text, flags=re.DOTALL)
        svg_blocks = re.findall(r'<svg\b.*?</svg>', text, re.DOTALL)
    
        if svg_blocks:
            tmp = re.findall(r'<svg\b.*?', svg_blocks[-1], re.DOTALL)
            if len(tmp) > 1:
                tmp2 = svg_blocks[-1].split('<svg')
                return '<svg ' + tmp2[-1]
            else:
                return svg_blocks[-1]
        else:
            if "<svg" in text and "</svg>" not in text:
                text += "</svg>"
                return text
            return default_svg
    
    @staticmethod
    def svg_conversion_check(topic, base_svg_code, default_svg):
        try:
            cairosvg.svg2png(bytestring=base_svg_code.encode('utf-8'), write_to="temp.png")
            return base_svg_code
        except Exception as e:
            print(f"Failed to convert {topic} due to {str(e)}, Returning default SVG.")
            return default_svg


class Model:
    def __init__(self):
        
        self.model_path = "./lora/lora_16bit_merged_3b_r64_s1000_i1000_v1"

        self.model = LLM(
            model = "./lora/lora_16bit_merged_3b_r64_s1000_i1000_v1",
            dtype = "float16",  
            max_model_len=2048,  
            gpu_memory_utilization=0.85 
        )

       
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model 
        gc.collect()     
    
    def get_response(self, description):

        #alpaca prompt
        instruction = """Generate SVG code to visually represent the following text description, while respecting the given constraints.
                <constraints>
                * **Allowed Elements:** `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`
                * **Allowed Attributes:** `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`
                </constraints>
                
                <example>
                    <description>"A red circle with a blue square inside"</description>
                    
                    ```svg
                    <svg viewBox="0 0 256 256" width="256" height="256">
                      <circle cx="50" cy="50" r="40" fill="red"/>
                      <rect x="30" y="30" width="40" height="40" fill="blue"/>
                      <...>
                       ...
                      <...>
                    </svg>
                ```
                </example>        
                
                Please ensure that the generated SVG code is well-formed, valid, and strictly adheres to these constraints.
                Focus on a clear and concise representation of the input description within the given limitations. 
                Always give the complete SVG code with nothing omitted. Never use an ellipsis.
                Do not include unnecessary explanations. Just give the code.
                """
            
        alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
            
                ### Instruction:
                {}
            
                ### Input:             
                <description>"{}"</description>
            
                ### Response:
                """

        formatted_input = alpaca_prompt.format(instruction, description)
        sampling_params = SamplingParams(temperature=0.5, top_p=0.95,max_tokens=1024)
        outputs = self.model.generate([formatted_input], sampling_params)
        
        #suitable for batch inputs as well        
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
        return generated_text
    
    def predict(self, description: str, max_new_tokens=2048) -> str:
        output_decoded = self.get_response(description)
        base_svg_code = SVGProcessor.clean_and_extract_svgs(output_decoded, self.default_svg)
        clean_svg_code = self.sanitizer.enforce_constraints(base_svg_code)
        return SVGProcessor.svg_conversion_check(description, clean_svg_code, self.default_svg)


In [20]:
model=Model()

WARNING 04-06 15:16:47 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 04-06 15:16:47 [config.py:585] This model supports multiple tasks: {'generate', 'classify', 'reward', 'score', 'embed'}. Defaulting to 'generate'.
INFO 04-06 15:16:47 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-06 15:16:47 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/lora_16bit_merged_3b_r64_s1000_i1000_v1', speculative_config=None, tokenizer='./lora/lora_16bit_merged_3b_r64_s1000_i1000_v1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_bac

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 04-06 15:16:50 [loader.py:447] Loading weights took 2.05 seconds
INFO 04-06 15:16:50 [gpu_model_runner.py:1186] Model loading took 6.0160 GB and 2.213933 seconds
INFO 04-06 15:16:56 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/2d3e90b0c6/rank_0_0 for vLLM's torch.compile
INFO 04-06 15:16:56 [backends.py:425] Dynamo bytecode transform time: 5.95 s
INFO 04-06 15:16:57 [backends.py:115] Directly load the compiled graph for shape None from the cache
INFO 04-06 15:17:01 [monitor.py:33] torch.compile takes 5.95 s in total
INFO 04-06 15:17:02 [kv_cache_utils.py:566] GPU KV cache size: 19,552 tokens
INFO 04-06 15:17:02 [kv_cache_utils.py:569] Maximum concurrency for 2,048 tokens per request: 9.55x
INFO 04-06 15:17:15 [gpu_model_runner.py:1534] Graph capturing finished in 14 secs, took 0.43 GiB
INFO 04-06 15:17:16 [core.py:151] init engine (profile, create kv cache, warmup model) took 25.12 seconds


In [21]:
model.predict('sun rising in the east')

Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.60s/it, est. speed input: 149.11


'<svg viewBox="0 0 256 256" width="256" height="256"><defs><radialGradient id="sunGradient" cx="0.5" cy="0.5" r="0.5"><stop offset="0%" stop-color="yellow"/><stop offset="100%" stop-color="orange"/></radialGradient><radialGradient id="skyGradient" cx="0.5" cy="0.5" r="0.5"><stop offset="0%" stop-color="blue"/><stop offset="100%" stop-color="lightblue"/></radialGradient></defs><g transform="rotate(45 128 128)"><circle cx="128" cy="128" r="30" fill="url(#sunGradient)"/><rect x="0" y="100" width="256" height="56" fill="url(#skyGradient)"/></g></svg>'

In [22]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/train.csv',header=[0])

In [23]:
from tqdm import tqdm
tqdm.pandas()
df['svg'] = df['description'].progress_apply(lambda x: model.predict(x))

  0%|                                                    | 0/15 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:10<00:00, 10.02s/it, est. speed input: 38.62 
 13%|█████▊                                      | 2/15 [00:10<01:05,  5.02s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.15s/it, est. speed input: 93.92 
 20%|████████▊                                   | 3/15 [00:14<00:55,  4.66s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.08s/it, est. speed input: 95.16 
 27%|███████████▋                                | 4/15 [00:18<00:48,  4.45s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.97s/it, est. speed input: 66.04 
 33%|██████████████▋                    

In [24]:
#write csv file for new metric score
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = re.sub(r'[^a-zA-Z0-9]', '_', model.model_path)
df.to_csv(f'./io_files/pred_{file_name}_{timestamp}.csv', index=False)

In [25]:
model.close_model()

In [34]:
#SigLip Score
evaluator = SVGMetricEvaluator()
df['sl_score'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg']), axis=1)


Using device: cuda


100%|███████████████████████████████████████████| 15/15 [00:00<00:00, 16.88it/s]


In [35]:
df['sl_score'].mean()

0.171867958361842